# DNR Main-Adaptive Finetuning on Generated HDF5 Data

This notebook integrates the new `dnr_diagnostic.h5` dataset with an adaptive DNR model workflow.

If a future `main_adative.py` / `main_adaptive.py` module or checkpoint is present, the adapter cell attempts to load it. If not, the notebook uses a compatible Graph Transformer fallback so the finetuning pipeline is executable and testable immediately.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════
DIAG_FILE = 'dnr_diagnostic.h5'
MAIN_ADATIVE_MODULE_CANDIDATES = ['main_adative.py', 'main_adaptive.py']
MAIN_ADATIVE_CHECKPOINT = 'main_adative.pt'      # optional input checkpoint
FINETUNED_CHECKPOINT = 'main_adative_finetuned.pt'
FINETUNE_METRICS_KEY = 'main_adative_finetune_metrics'

seed = 42
batch_size = 32
finetune_epochs = 20
learning_rate = 2e-4
weight_decay = 1e-4
w_bce = 1.0
w_loss = 0.25
w_voltage = 1.0
validation_fraction = 0.20
V_MIN_PU = 0.90
N_BUS = 33
N_LINES = 37
N_OPEN = 5
DEVICE = 'cuda'

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════════════
import ast
import importlib.util
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, global_mean_pool

try:
    display
except NameError:
    def display(obj):
        print(obj)

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() and DEVICE == 'cuda' else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# LOAD NEW HDF5 DATASET
# ═══════════════════════════════════════════════════════════════════
if not Path(DIAG_FILE).exists():
    raise FileNotFoundError(f'{DIAG_FILE} not found. Run 01_DNR_Data_Generation.ipynb first.')

with pd.HDFStore(DIAG_FILE, mode='r') as store:
    print('HDF5 keys:', store.keys())
    scenarios_df = store['scenarios']
    summary_df = store['summary'] if '/summary' in store.keys() else pd.DataFrame()

print('Scenarios:', scenarios_df.shape)
display(summary_df.head())

p_cols = [f'p_bus_{b}' for b in range(1, N_BUS)]
q_cols = [f'q_bus_{b}' for b in range(1, N_BUS)]
missing = [c for c in p_cols + q_cols + ['topo', 'loss_mw', 'v_min_pu'] if c not in scenarios_df.columns]
if missing:
    raise KeyError(f'Missing required columns: {missing}')

print('Scenario health counts:')
if 'scenario_health' in scenarios_df.columns:
    display(scenarios_df['scenario_health'].value_counts(dropna=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PYG DATASET ADAPTER FOR NEW HDF5 SCHEMA
# ═══════════════════════════════════════════════════════════════════
FROM_BUS = np.array([
    1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,
    2,19,20,21,3,23,24,6,26,27,28,29,30,31,32,
    21,9,12,18,25
], dtype=np.int64) - 1
TO_BUS = np.array([
    2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,
    19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,
    8,15,22,33,29
], dtype=np.int64) - 1
R_OHM = np.array([
    0.0922,0.4930,0.3660,0.3811,0.8190,0.1872,1.7114,1.0300,1.0440,
    0.1966,0.3744,1.4680,0.5416,0.5910,0.7463,1.2890,0.7320,
    0.1640,1.5042,0.4095,0.7089,0.4512,0.8980,0.8960,0.2030,0.2842,
    1.0590,0.8042,0.5075,0.9744,0.3105,0.3410,2.0000,2.0000,2.0000,0.5000,0.5000
], dtype=np.float32)
X_OHM = np.array([
    0.0470,0.2511,0.1864,0.1941,0.7070,0.6188,1.2351,0.7400,0.7400,
    0.0650,0.1238,1.1550,0.7129,0.5260,0.5450,1.7210,0.5740,
    0.1565,1.3554,0.4784,0.9373,0.3083,0.7091,0.7011,0.1034,0.1447,
    0.9337,0.7006,0.2585,0.9630,0.3619,0.5302,2.0000,2.0000,2.0000,0.5000,0.5000
], dtype=np.float32)

edge_index = torch.tensor(np.stack([FROM_BUS, TO_BUS]), dtype=torch.long)
edge_attr = torch.tensor(np.stack([
    R_OHM / (R_OHM.max() + 1e-9),
    X_OHM / (X_OHM.max() + 1e-9),
    (np.arange(N_LINES) >= 32).astype(np.float32),
    np.arange(N_LINES, dtype=np.float32) / (N_LINES - 1),
], axis=1), dtype=torch.float32)

season_vocab = ['winter', 'summer', 'spring', 'autumn', 'steered', 'stress']
season_to_idx = {s: i for i, s in enumerate(season_vocab)}
p_scale = max(float(scenarios_df[p_cols].to_numpy().max()), 1e-6)
q_scale = max(float(scenarios_df[q_cols].to_numpy().max()), 1e-6)
loss_mu = float(scenarios_df['loss_mw'].mean())
loss_sigma = float(scenarios_df['loss_mw'].std() + 1e-6)

bus_norm = np.arange(N_BUS, dtype=np.float32) / (N_BUS - 1)
slack_flag = (np.arange(N_BUS) == 0).astype(np.float32)

def parse_topology(value):
    return np.array(ast.literal_eval(str(value)), dtype=np.int64)

def infer_task(row):
    season = str(row.get('season', ''))
    if season == 'steered': return 'steered'
    if season == 'stress': return 'stress'
    return 'realistic'

class DNRFinetuneDataset(Dataset):
    def __init__(self, frame):
        super().__init__()
        self.frame = frame.reset_index(drop=True).copy()
        self.task_labels = [infer_task(row) for _, row in self.frame.iterrows()]
        self.task_to_indices = defaultdict(list)
        for i, task in enumerate(self.task_labels):
            self.task_to_indices[task].append(i)

    def len(self):
        return len(self.frame)

    def get(self, idx):
        row = self.frame.iloc[int(idx)]
        p = np.zeros(N_BUS, dtype=np.float32)
        q = np.zeros(N_BUS, dtype=np.float32)
        p[1:] = row[p_cols].to_numpy(dtype=np.float32)
        q[1:] = row[q_cols].to_numpy(dtype=np.float32)
        hour = float(row.get('hour', -1))
        hour_norm = 0.0 if hour < 0 else hour / 23.0
        flags = np.tile(np.array([
            hour_norm,
            1.0 if hour < 0 else 0.0,
            float(bool(row.get('weekend', False))),
            float(bool(row.get('ev_active', False))),
            float(bool(row.get('pv_active', False))),
            float(bool(row.get('default_voltage_violation', False))),
            float(bool(row.get('recovered_voltage_violation', False))),
        ], dtype=np.float32), (N_BUS, 1))
        season_onehot = np.zeros(len(season_vocab), dtype=np.float32)
        season = str(row.get('season', ''))
        if season in season_to_idx:
            season_onehot[season_to_idx[season]] = 1.0
        season_rep = np.tile(season_onehot, (N_BUS, 1))
        x = np.column_stack([p / p_scale, q / q_scale, bus_norm, slack_flag, flags, season_rep]).astype(np.float32)
        y = torch.zeros(N_LINES, dtype=torch.float32)
        y[torch.tensor(parse_topology(row['topo']) - 1, dtype=torch.long)] = 1.0
        return Data(
            x=torch.tensor(x, dtype=torch.float32),
            edge_index=edge_index.clone(),
            edge_attr=edge_attr.clone(),
            y=y,
            loss_target=torch.tensor([float(row['loss_mw'])], dtype=torch.float32),
            loss_norm_target=torch.tensor([(float(row['loss_mw']) - loss_mu) / loss_sigma], dtype=torch.float32),
            vmin_target=torch.tensor([float(row['v_min_pu'])], dtype=torch.float32),
            scenario_idx=torch.tensor([idx], dtype=torch.long),
        )

dataset = DNRFinetuneDataset(scenarios_df)
print('Dataset size:', len(dataset))
print('Node features:', tuple(dataset[0].x.shape), 'Edge attr:', tuple(dataset[0].edge_attr.shape))
print('Tasks:', {k: len(v) for k, v in dataset.task_to_indices.items()})

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# OPTIONAL MAIN_ADATIVE MODEL LOADER + COMPATIBLE FALLBACK
# ═══════════════════════════════════════════════════════════════════
def try_import_main_adative():
    for candidate in MAIN_ADATIVE_MODULE_CANDIDATES:
        path = Path(candidate)
        if not path.exists():
            continue
        spec = importlib.util.spec_from_file_location(path.stem, path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        print(f'Imported optional module: {candidate}')
        return mod
    print('No main_adative/main_adaptive module found; using compatible fallback model.')
    return None

class CompatibleAdaptiveGT(nn.Module):
    def __init__(self, node_dim, edge_dim=4, hidden=96, heads=4):
        super().__init__()
        self.node_enc = nn.Linear(node_dim, hidden)
        self.edge_enc = nn.Linear(edge_dim, hidden)
        self.conv1 = TransformerConv(hidden, hidden // heads, heads=heads, edge_dim=hidden, dropout=0.1)
        self.conv2 = TransformerConv(hidden, hidden // heads, heads=heads, edge_dim=hidden, dropout=0.1)
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)
        self.edge_head = nn.Sequential(nn.Linear(2 * hidden + hidden, hidden), nn.ReLU(), nn.Dropout(0.1), nn.Linear(hidden, 1))
        self.loss_head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.vmin_head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, data):
        h = torch.relu(self.node_enc(data.x))
        e = torch.relu(self.edge_enc(data.edge_attr))
        h = self.norm1(h + torch.relu(self.conv1(h, data.edge_index, e)))
        h = self.norm2(h + torch.relu(self.conv2(h, data.edge_index, e)))
        src, dst = data.edge_index
        # PyG batches shift edge_index per graph, so modulo maps back to edge ids for edge features.
        edge_ids = torch.arange(src.numel(), device=h.device) % N_LINES
        edge_logits = self.edge_head(torch.cat([h[src], h[dst], e[edge_ids]], dim=-1)).squeeze(-1)
        graph_h = global_mean_pool(h, data.batch)
        loss_norm = self.loss_head(graph_h).squeeze(-1)
        vmin = 0.75 + 0.35 * torch.sigmoid(self.vmin_head(graph_h).squeeze(-1))
        return edge_logits.view(-1, N_LINES), loss_norm, vmin

def build_or_load_model(node_dim):
    mod = try_import_main_adative()
    model = None
    if mod is not None:
        for builder_name in ['build_model', 'create_model', 'get_model']:
            if hasattr(mod, builder_name):
                try:
                    model = getattr(mod, builder_name)(node_dim=node_dim, edge_dim=edge_attr.shape[1], n_lines=N_LINES)
                    print(f'Built model using {builder_name}()')
                    break
                except TypeError:
                    pass
        for class_name in ['MainAdaptive', 'MainAdative', 'AdaptiveGT', 'MetaPhyGT']:
            if model is None and hasattr(mod, class_name):
                try:
                    model = getattr(mod, class_name)(node_dim=node_dim)
                    print(f'Built model using {class_name}')
                    break
                except TypeError:
                    pass
    if model is None:
        model = CompatibleAdaptiveGT(node_dim=node_dim)
    if Path(MAIN_ADATIVE_CHECKPOINT).exists():
        state = torch.load(MAIN_ADATIVE_CHECKPOINT, map_location='cpu')
        state_dict = state.get('state_dict', state) if isinstance(state, dict) else state
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        print(f'Loaded checkpoint {MAIN_ADATIVE_CHECKPOINT}: missing={len(missing)}, unexpected={len(unexpected)}')
    return model.to(device)

model = build_or_load_model(dataset[0].x.shape[1])
print(model.__class__.__name__)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FINETUNING LOOP ON NEW HDF5 DATA
# ═══════════════════════════════════════════════════════════════════
indices = np.arange(len(dataset))
np.random.default_rng(seed).shuffle(indices)
val_n = max(1, int(len(indices) * validation_fraction))
val_idx = indices[:val_n].tolist()
train_idx = indices[val_n:].tolist()

loader_train = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True)
loader_val = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size, shuffle=False)

train_y = torch.stack([dataset[i].y for i in train_idx])
pos = train_y.sum(dim=0)
neg = len(train_idx) - pos
pos_weight = (neg / pos.clamp_min(1.0)).clamp(1.0, 50.0).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

loss_sigma_t = torch.tensor(loss_sigma, dtype=torch.float32, device=device)
loss_mu_t = torch.tensor(loss_mu, dtype=torch.float32, device=device)

def batch_loss(batch):
    batch = batch.to(device)
    logits, pred_loss_norm, pred_vmin = model(batch)
    true_edges = batch.y.view(-1, N_LINES)
    true_loss_norm = batch.loss_norm_target.view(-1)
    bce = F.binary_cross_entropy_with_logits(logits, true_edges, pos_weight=pos_weight)
    loss_mse = F.mse_loss(pred_loss_norm, true_loss_norm)
    volt_pen = torch.relu(V_MIN_PU - pred_vmin).pow(2).mean()
    total = w_bce * bce + w_loss * loss_mse + w_voltage * volt_pen
    return total, bce.detach(), loss_mse.detach(), volt_pen.detach()

def evaluate(loader):
    model.eval()
    totals=[]; exact=[]; overlap=[]; loss_mae=[]; v_mae=[]
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits, pred_loss_norm, pred_vmin = model(batch)
            true_edges = batch.y.view(-1, N_LINES)
            pred_open = torch.zeros_like(true_edges)
            topk = torch.topk(logits, k=N_OPEN, dim=1).indices
            pred_open.scatter_(1, topk, 1.0)
            exact.append((pred_open == true_edges).all(dim=1).float().cpu())
            overlap.append((pred_open * true_edges).sum(dim=1).div(N_OPEN).cpu())
            pred_loss = pred_loss_norm * loss_sigma_t + loss_mu_t
            loss_mae.append((pred_loss - batch.loss_target.view(-1)).abs().cpu())
            v_mae.append((pred_vmin - batch.vmin_target.view(-1)).abs().cpu())
            totals.append(batch_loss(batch)[0].detach().cpu())
    return {
        'loss': float(torch.stack(totals).mean()),
        'exact_topology': float(torch.cat(exact).mean()),
        'switch_overlap': float(torch.cat(overlap).mean()),
        'loss_mae_mw': float(torch.cat(loss_mae).mean()),
        'vmin_mae_pu': float(torch.cat(v_mae).mean()),
    }

history=[]
for epoch in range(1, finetune_epochs + 1):
    model.train()
    batch_losses=[]
    for batch in loader_train:
        optimizer.zero_grad()
        total, bce, loss_mse, volt_pen = batch_loss(batch)
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        batch_losses.append(float(total.detach().cpu()))
    val_metrics = evaluate(loader_val)
    row = {'epoch': epoch, 'train_loss': float(np.mean(batch_losses)), **{f'val_{k}': v for k, v in val_metrics.items()}}
    history.append(row)
    print(f"Epoch {epoch:03d}/{finetune_epochs}: train={row['train_loss']:.4f} val={row['val_loss']:.4f} overlap={row['val_switch_overlap']:.3f}")

history_df = pd.DataFrame(history)
torch.save({'state_dict': model.state_dict(), 'node_dim': dataset[0].x.shape[1], 'history': history}, FINETUNED_CHECKPOINT)
history_df.to_hdf(DIAG_FILE, key=FINETUNE_METRICS_KEY, mode='a')
print(f'Saved checkpoint: {FINETUNED_CHECKPOINT}')
print(f'Saved metrics to {DIAG_FILE}:{FINETUNE_METRICS_KEY}')
display(history_df.tail())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# QUICK POST-FINETUNE SUMMARY
# ═══════════════════════════════════════════════════════════════════
print('═' * 70)
print('MAIN_ADATIVE FINETUNE SUMMARY')
print('═' * 70)
print(f'Dataset rows          : {len(dataset)}')
print(f'Train / validation    : {len(train_idx)} / {len(val_idx)}')
print(f'Final validation loss : {history_df.val_loss.iloc[-1]:.5f}')
print(f'Switch overlap        : {history_df.val_switch_overlap.iloc[-1]*100:.1f}%')
print(f'Exact topology        : {history_df.val_exact_topology.iloc[-1]*100:.1f}%')
print(f'Loss MAE              : {history_df.val_loss_mae_mw.iloc[-1]:.6f} MW')
print(f'Vmin MAE              : {history_df.val_vmin_mae_pu.iloc[-1]:.6f} pu')
print('═' * 70)